In [2]:
%pip install graphviz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from graphviz import Digraph

# UML-style record-based diagram for Terraform GCP Infrastructure (Gitea Deployment)
dot = Digraph("Terraform_GCP_UML", format="png")
dot.attr(rankdir="TB", fontsize="11", fontname="Arial")
dot.attr("node", shape="record", style="filled", fillcolor="#E1F5FE", fontname="Arial")

# Input Variables (just referenced, not drawn as a box)
input_vars = [
    "network_name", "region", "cidr_block", "project_id", 
    "gitea_vm_name", "machine_type", "zone/image", "instance_tags", "bucket_name"
]

# gcp_network Module
dot.node("gcp_network", '''{
    gcp_network |
    {Input Variables|network_name\\lregion\\lcidr_block\\lproject_id\\l} |
    {Output|vpc_network_id\\lsubnet_public_self_link\\l}
}''')

# compute_engine Module
dot.node("compute_engine", '''{
    compute_engine |
    {Input Variables|gitea_vm_name\\lmachine_type\\lzone/image\\linstance_tags\\lvpc_network_id\\lsubnet_public_self_link\\l} |
    {Output|gitea_instance_public_ip\\l}
}''')

# firewall Module
dot.node("firewall", '''{
    firewall |
    {Input Variables|network_name\\linstance_tags\\l} |
    {Output|firewall_rules_configured\\l}
}''')

# gcs_bucket Module
dot.node("gcs_bucket", '''{
    gcs_bucket |
    {Input Variables|bucket_name\\lproject_id\\l} |
    {Output|gcs_bucket_name\\lgcs_bucket_url\\l}
}''')

# Outputs container node (optional grouping)
dot.node("outputs", '''{
    Outputs |
    {Exported Values|vpc_network_id\\lsubnet_public_self_link\\lgitea_instance_public_ip\\lgcs_bucket_name\\lgcs_bucket_url\\l}
}''', fillcolor="#FFF3E0")

# Arrows showing flow and dependencies
dot.edge("gcp_network", "compute_engine", label="provides subnet & vpc")
dot.edge("gcp_network", "firewall", label="provides network")
dot.edge("gcp_network", "gcs_bucket", style="dashed", label="project_id")

dot.edge("compute_engine", "outputs", label="public_ip")
dot.edge("gcp_network", "outputs", label="vpc & subnet")
dot.edge("gcs_bucket", "outputs", label="bucket info")
dot.edge("firewall", "outputs", style="dashed", label="rules configured")

# Render and view the file
output_path = "./terraform_gcp_uml"
dot.render(output_path, cleanup=False)
output_path + ".png"


'./terraform_gcp_uml.png'